In [ ]:
#Importacao das bibliotecas
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

#Definicao do que será necessário nos campos name e password e o tamanho minimo
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

#Os tipos de roles que irao existir e suas respectivas atribuicoes, quando for dito que a role é 1, o user é um author
class Role (enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

#Definindo o usuario e todos os campos que o mesmo terá, com definicao, exemplo e se é multavel ou nao 
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples = ["user@arjancodes.com"],
        description ="The email of the user",
        frozen = True
    )
    password: SecretStr = Field(
        examples = ["Password123"], description = "The password of the user"
    )
    role: Role = Field(default = None, description = "The role of the user")

#Feito a validacao de cada campo separadamente, usando da funco field_validator
#Validacao do campo "name"
    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v): #Verificando se esta de acordo com o que foi definido para o campo nome
            raise ValueError(
                "Name is invalid, must contain only letters and be at last 2 characters long" #Caso esteja fora do padrao, retornara essa mensagem
            )
        return v  

#Validacao do campo role, ira verificar se o campo recebe um dos valores estipulados para as roles, como 1,2,4 ...
    @field_validator("role", mode = "before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f'Role is invalid, please use one of the following: {", ".join([x.name for x in Role])}' #Se nao receber um valor dentro do esperado, retorara essa mensagem
            )

#Validacao do campo senha, a qual ocorre depois pois necessita do campo "name" para fazer sua verificacao
    @model_validator(mode = "before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:   #Verifica se os campos "name" e "password" foram preenchidos pois sao obrigatorios
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold(): #Verifica se o "nome" esta presente na senha, caso esteja é retornado erro e informado que nao pode conter
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]): #Verifica o formato da senha para estar dentro do esperado, caso nao seja o esperado, retorna a mensagem informando o que se espera da senha
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

#Faz a verificacao da existencia dos campos, se estao corretos e cria um objeto para o user
def validate (data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        print(e)

#Funcao main que informa o que se espera dos dados com "good_data" e mostra possiveis erros com "bad_data"
def main() -> None:
    test_data = dict(
        #O que se espera de um preenchimento correto
        good_data = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin"
        },
        #Erro de preenchimento do campo "role"
        bad_role = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer"
        },
        #Erro de preenchimento do campo "password"
        bad_data = {
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "bad password",
        },
        #Erro de preenchimento do campo "name"
        bad_name = {
            "name": "Arjan >-<",
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        #Erro de preenchimento do campo "password" com o nome presente nele
        duplicate ={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123",
        },
        #Erro de preenchimento por falta de dados
        missing_data = {
            "email": "<bad data>",
            "password": "<bad data>",
        }
    )